# EEG Eye State Classification

Goal: predict whether a person's eyes are open or closed from 14 EEG sensor measurements.

Dataset: UCI Machine Learning Repository, EEG Eye State, dataset ID 264.

Models:
1. Logistic Regression
2. Random Forest

Evaluation:
Accuracy, precision, recall, F1 score, confusion matrix, and feature importance.

In [ ]:
%pip install -q ucimlrepo

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd

from ucimlrepo import fetch_ucirepo
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 1. Load the public dataset

In [ ]:
eeg_eye_state = fetch_ucirepo(id=264)

X = eeg_eye_state.data.features.copy()
y = eeg_eye_state.data.targets.squeeze().astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts().sort_index().rename(index={0: "Eyes open", 1: "Eyes closed"}))

X.head()

Target definition:

0 = eyes open

1 = eyes closed

In [ ]:
class_counts = y.value_counts().sort_index()
class_counts.index = ["Eyes open", "Eyes closed"]

ax = class_counts.plot(kind="bar", rot=0, title="Class Distribution")
ax.set_xlabel("Eye state")
ax.set_ylabel("Number of observations")
plt.tight_layout()
plt.show()

## 2. Split the data

A stratified 80/20 split preserves the class ratio in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

## 3. Train two baseline models

In [ ]:
logistic_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

random_forest_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

models = {
    "Logistic Regression": logistic_pipeline,
    "Random Forest": random_forest_pipeline,
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append(
        {
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        }
    )

results_df = pd.DataFrame(results).sort_values("F1 Score", ascending=False)
results_df.round(4)

## 4. Evaluate the best model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
best_predictions = predictions[best_model_name]

print("Best model:", best_model_name)
print()
print(classification_report(
    y_test,
    best_predictions,
    target_names=["Eyes open", "Eyes closed"],
    digits=4,
))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_predictions,
    display_labels=["Eyes open", "Eyes closed"],
)
plt.title(f"Confusion Matrix: {best_model_name}")
plt.tight_layout()
plt.show()

## 5. Inspect Random Forest feature importance

In [ ]:
rf_model = random_forest_pipeline.named_steps["model"]

feature_importance = (
    pd.Series(rf_model.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
)

feature_importance.to_frame("Importance").head(10)

In [ ]:
ax = feature_importance.head(10).sort_values().plot(
    kind="barh",
    title="Top 10 EEG Features by Random Forest Importance",
)
ax.set_xlabel("Importance")
ax.set_ylabel("EEG feature")
plt.tight_layout()
plt.show()

## Conclusion

This notebook creates a reproducible baseline for EEG eye-state classification.

Main limitation:

The observations come from one continuous chronological EEG recording. A random train-test split can overestimate real-world performance because nearby observations may be highly similar.

Next improvement:

Use a chronological split or time-series cross-validation and compare the result with the random split.